# 02 Baseline and Ranking Models

This notebook trains baseline models, produces ranking scores per cell line, and evaluates prioritization metrics.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score
from lightgbm import LGBMClassifier, LGBMRanker
from scipy.sparse import hstack

from gbm_pipeline import (
    RANDOM_SEED,
    build_categorical_matrix,
    build_multihot_matrix,
    build_structure_matrix,
    define_hit_label,
    load_ctd_mapping,
    map_at_k,
    ndcg_at_k,
    precision_at_k,
    recall_at_k,
)

output_dir = Path('outputs')
output_dir.mkdir(exist_ok=True)

df = pd.read_csv(output_dir / 'hts_prepared.csv')

# Target label using the rule-based definition
if 'label_rule' not in df.columns:
    df['label_rule'] = define_hit_label(df)

label_col = 'label_rule'

print(df.shape)
print(df[label_col].value_counts())

## Train/Test Split

Group split by structure to reduce compound leakage across train/test.

In [ ]:
groups = df['Structure'].astype('string').fillna('unknown')

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(df, df[label_col], groups=groups))

df_train = df.iloc[train_idx].copy()
df_test = df.iloc[test_idx].copy()

y_train = df_train[label_col].values
y_test = df_test[label_col].values

print(df_train.shape, df_test.shape)

## Build CTD Mappings

Reuse CTD mapping tables for expanded annotation features.

In [ ]:
ctd_pathways = load_ctd_mapping('CTD mapping/CTD_chem_pathways_enriched_1535.tsv', 'PathwayName')
ctd_genes = load_ctd_mapping('CTD mapping/CTD_chem_cgixns_1535.tsv', 'GeneSymbol')
ctd_diseases = load_ctd_mapping('CTD mapping/CTD_chem_diseases_1535.tsv', 'DiseaseID')
ctd_go = load_ctd_mapping('CTD mapping/CTD_chem_go_enriched_1535.tsv', 'GoTermID')
ctd_phenotypes = load_ctd_mapping('CTD mapping/CTD_chem_phenotypes_curated_1535.tsv', 'PhenotypeID')

## Feature Builders

Prepare structure, categorical, and annotation features.

In [ ]:
def build_feature_set(df_subset, vectorizer=None, ohe=None, feature_set='structure_only'):
    X_struct, vectorizer, struct_names, struct_method = build_structure_matrix(
        df_subset['Structure'].astype('string').fillna(''),
        vectorizer=vectorizer,
    )

    X_cat, ohe, cat_names = build_categorical_matrix(df_subset, ohe=ohe, fit=ohe is None)
    matrices = [X_struct, X_cat]
    feature_names = struct_names + cat_names

    ctd_names = df_subset['ctd_name_norm'].tolist()
    if feature_set in ['structure_pathway', 'structure_expanded']:
        X_path, path_names = build_multihot_matrix(ctd_names, ctd_pathways, 'pathway')
        matrices.append(X_path)
        feature_names += path_names

    if feature_set == 'structure_expanded':
        X_gene, gene_names = build_multihot_matrix(ctd_names, ctd_genes, 'gene')
        X_dis, dis_names = build_multihot_matrix(ctd_names, ctd_diseases, 'disease')
        X_go, go_names = build_multihot_matrix(ctd_names, ctd_go, 'go')
        X_pheno, pheno_names = build_multihot_matrix(ctd_names, ctd_phenotypes, 'phenotype')
        matrices += [X_gene, X_dis, X_go, X_pheno]
        feature_names += gene_names + dis_names + go_names + pheno_names

    X = hstack(matrices).tocsr()
    return X, vectorizer, ohe, feature_names, struct_method

## Model Training and Evaluation

Train classifiers for each feature set and evaluate ranking metrics per cell line.

In [ ]:
def evaluate_per_cell_line(df_eval, score_col, label_col, k_values):
    rows = []
    for cell_line, group in df_eval.groupby('Cell_line'):
        labels = group[label_col].values.astype(int)
        scores = group[score_col].values.astype(float)
        for k in k_values:
            rows.append({
                'cell_line': cell_line,
                'k': k,
                'precision_at_k': precision_at_k(labels, scores, k),
                'recall_at_k': recall_at_k(labels, scores, k),
                'ndcg_at_k': ndcg_at_k(labels, scores, k),
                'map_at_k': map_at_k(labels, scores, k),
            })
    return pd.DataFrame(rows)

In [ ]:
feature_sets = ['structure_only', 'structure_pathway', 'structure_expanded']

metrics_rows = []
rank_rows = []

vectorizer = None
ohe = None

for feature_set in feature_sets:
    X_train, vectorizer, ohe, feature_names, struct_method = build_feature_set(
        df_train, vectorizer=vectorizer, ohe=ohe, feature_set=feature_set
    )
    X_test, _, _, _, _ = build_feature_set(
        df_test, vectorizer=vectorizer, ohe=ohe, feature_set=feature_set
    )

    model = LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=64,
        class_weight='balanced',
        random_state=RANDOM_SEED,
    )
    model.fit(X_train, y_train)
    preds = model.predict_proba(X_test)[:, 1]

    roc = roc_auc_score(y_test, preds)
    pr = average_precision_score(y_test, preds)

    metrics_rows.append({
        'feature_set': feature_set,
        'structure_method': struct_method,
        'model': 'lightgbm_classifier',
        'roc_auc': roc,
        'pr_auc': pr,
    })

    df_rank = df_test[['Sample ID', 'Structure', 'Sample Name', 'Cell_line', 'ctd_name_norm', label_col]].copy()
    df_rank['score'] = preds
    df_rank['feature_set'] = feature_set
    df_rank['model'] = 'lightgbm_classifier'
    rank_rows.append(df_rank)

metrics_df = pd.DataFrame(metrics_rows)
metrics_df

## LambdaRank (Learning-to-Rank)

Train a LightGBM ranker per feature set using cell line as the query group.

In [ ]:
ranker_rows = []

for feature_set in feature_sets:
    # Sort by cell line to define query groups
    df_train_rank = df_train.sort_values('Cell_line').reset_index(drop=True)
    df_test_rank = df_test.sort_values('Cell_line').reset_index(drop=True)

    X_train, vectorizer_r, ohe_r, feature_names, struct_method = build_feature_set(
        df_train_rank, vectorizer=None, ohe=None, feature_set=feature_set
    )
    X_test, _, _, _, _ = build_feature_set(
        df_test_rank, vectorizer=vectorizer_r, ohe=ohe_r, feature_set=feature_set
    )

    y_train_rank = df_train_rank[label_col].values.astype(int)
    y_test_rank = df_test_rank[label_col].values.astype(int)

    group_train = df_train_rank.groupby('Cell_line').size().tolist()
    group_test = df_test_rank.groupby('Cell_line').size().tolist()

    ranker = LGBMRanker(
        objective='lambdarank',
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=64,
        random_state=RANDOM_SEED,
    )

    ranker.fit(
        X_train,
        y_train_rank,
        group=group_train,
        eval_set=[(X_test, y_test_rank)],
        eval_group=[group_test],
        eval_at=[10, 25, 50],
    )

    preds = ranker.predict(X_test)

    df_rank = df_test_rank[['Sample ID', 'Structure', 'Sample Name', 'Cell_line', 'ctd_name_norm', label_col]].copy()
    df_rank['score'] = preds
    df_rank['feature_set'] = feature_set
    df_rank['model'] = 'lambdarank'
    ranker_rows.append(df_rank)

ranker_df = pd.concat(ranker_rows, ignore_index=True)
ranker_df.head()

In [ ]:
rank_df = pd.concat(rank_rows + [ranker_df], ignore_index=True)
rank_df.head()

In [ ]:
k_values = [10, 25, 50]
ranking_metrics = []

for feature_set in feature_sets:
    for model_name in ['lightgbm_classifier', 'lambdarank']:
        subset = rank_df[(rank_df['feature_set'] == feature_set) & (rank_df['model'] == model_name)].copy()
        per_cell = evaluate_per_cell_line(subset, 'score', label_col, k_values)
        per_cell['feature_set'] = feature_set
        per_cell['model'] = model_name
        ranking_metrics.append(per_cell)

ranking_metrics_df = pd.concat(ranking_metrics, ignore_index=True)
ranking_metrics_df.head()

## Aggregate Ranking Across Cell Lines

Aggregate by structure using mean score across cell lines. A compound is labeled positive if it is a hit in any test cell line.

In [ ]:
aggregate_rows = []

for feature_set in feature_sets:
    for model_name in ['lightgbm_classifier', 'lambdarank']:
        subset = rank_df[(rank_df['feature_set'] == feature_set) & (rank_df['model'] == model_name)].copy()
        grouped = subset.groupby('Structure')
        agg = grouped.agg(
            score_mean=('score', 'mean'),
            score_max=('score', 'max'),
            hit_any=(label_col, 'max'),
            n_cell_lines=('Cell_line', 'nunique'),
        ).reset_index()
        agg['feature_set'] = feature_set
        agg['model'] = model_name

        try:
            roc = roc_auc_score(agg['hit_any'], agg['score_mean'])
        except ValueError:
            roc = float('nan')
        try:
            pr = average_precision_score(agg['hit_any'], agg['score_mean'])
        except ValueError:
            pr = float('nan')

        aggregate_rows.append({
            'feature_set': feature_set,
            'model': model_name,
            'roc_auc': roc,
            'pr_auc': pr,
            'n_compounds': len(agg),
        })

        out_path = output_dir / f'aggregate_ranked_candidates_{feature_set}_{model_name}.csv'
        agg.sort_values('score_mean', ascending=False).to_csv(out_path, index=False)

aggregate_metrics_df = pd.DataFrame(aggregate_rows)
aggregate_metrics_df

## Save Outputs

Persist metrics and ranked candidate lists per cell line.

In [ ]:
metrics_df.to_csv(output_dir / 'baseline_metrics.csv', index=False)
ranking_metrics_df.to_csv(output_dir / 'ranking_metrics_per_cell_line.csv', index=False)
aggregate_metrics_df.to_csv(output_dir / 'aggregate_metrics.csv', index=False)
rank_df.to_csv(output_dir / 'rank_scores.csv', index=False)

# Ranked candidates per cell line and feature set
for feature_set in feature_sets:
    for model_name in ['lightgbm_classifier', 'lambdarank']:
        subset = rank_df[(rank_df['feature_set'] == feature_set) & (rank_df['model'] == model_name)].copy()
        for cell_line, group in subset.groupby('Cell_line'):
            out_path = output_dir / f'ranked_candidates_{cell_line}_{feature_set}_{model_name}.csv'
            group.sort_values('score', ascending=False).to_csv(out_path, index=False)

print('Saved outputs to', output_dir)